# ETL da camada Silver para camada Gold - Microsoft Security Incident Prediction

Este notebook realiza o ETL (Extract, Transform, Load) dos dados da camada Silver para a camada Gold. 
Focamos em manter apenas colunas relevantes para construção de dashboards futuros, como agregações temporais, geográficas e por severidade de incidentes.

## Colunas Mantidas para Dashboard
- timestamp: Para tendências temporais.
- orgid: Agregação por organização.
- detectorid: Detetores de alertas.
- alerttitle: Títulos de alertas.
- category: Categorias de incidentes.
- mitretechniques: Técnicas MITRE.
- incidentgrade: Severidade (target principal).
- entitytype: Tipos de entidades.
- evidencerole: Papel da evidência.
- osfamily: Família de SO.
- osversion: Versão de SO.
- lastverdict: Veredito final.
- countrycode, state, city: Localização geográfica.


## EXTRACT

Extraímos os dados do arquivo CSV da camada Silver.


In [3]:
import re
from pathlib import Path
import pandas as pd
import numpy as np
import warnings
from sqlalchemy import create_engine
import sqlalchemy

warnings.filterwarnings('ignore')

print("=== ETL Silver -> Gold (EXTRACT / TRANSFORM / LOAD) (Postgres-only) ===")

# caminhos (apenas para localizar docker-compose se houver)
repo_root = Path.cwd().parent if Path.cwd().name == 'transformer' else Path.cwd()


# ---------- Helpers ----------
def read_postgres_from_docker_compose(path: Path):
    """
    Lê credenciais básicas do docker-compose.yml (parsing simples).
    Retorna dict com chaves: user, password, db, host, port
    Se o arquivo não existir, retorna None.
    """
    if not path.exists():
        return None

    txt = path.read_text(encoding='utf-8')

    def find_var(k: str):
        # captura valor após a chave, com ou sem aspas
        pattern = rf'{re.escape(k)}\s*:\s*["\']?([^\n"\' ]+)'
        m = re.search(pattern, txt)
        return m.group(1).strip() if m else None

    # tenta chaves com case comum e lowercase
    user = find_var('POSTGRES_USER') or find_var('postgres_user') or 'postgres'
    password = find_var('POSTGRES_PASSWORD') or find_var('postgres_password') or 'postgres'
    db = find_var('POSTGRES_DB') or find_var('postgres_db') or find_var('POSTGRES_NAME') or 'microsoft-security'

    # busca mapeamento de portas no compose (ex: "5433:5432")
    mport = re.search(r'ports\s*:\s*\n\s*-\s*["\']?(\d+):\d+', txt, re.MULTILINE)
    port = mport.group(1) if mport else None

    # fallback para variável dedicada
    if not port:
        port = find_var('POSTGRES_PORT') or find_var('postgres_port')

    host = 'localhost'
    port = port or '5432'

    return dict(user=user, password=password, db=db, host=host, port=port)


def create_engine_from_compose(repo_root: Path):
    """
    Tenta criar uma engine SQLAlchemy lendo docker-compose.yml.
    Retorna (engine_or_None, pg_info_dict)
    """
    docker_path = repo_root / 'docker-compose.yml'
    pg = read_postgres_from_docker_compose(docker_path)
    if not pg:
        # se não encontrar compose, tentar variáveis de ambiente
        pg = {
            'user': os.getenv('POSTGRES_USER', 'postgres'),
            'password': os.getenv('POSTGRES_PASSWORD', 'postgres'),
            'db': os.getenv('POSTGRES_DB', 'microsoft-security'),
            'host': os.getenv('POSTGRES_HOST', 'localhost'),
            'port': os.getenv('POSTGRES_PORT', '5432')
        }

    url = f"postgresql+psycopg2://{pg['user']}:{pg['password']}@{pg['host']}:{pg['port']}/{pg['db']}"

    try:
        eng = create_engine(url)
        # testar conexão rapidamente
        with eng.connect() as conn:
            pass
        print(f"Connected to Postgres: {pg['host']}:{pg['port']}/{pg['db']}")
        return eng, pg
    except Exception as e:
        print("Postgres unavailable via SQLAlchemy. Exception:", e)
        return None, pg


# ---------- EXTRACT ----------
def extract_silver(engine):
    """
    Lê os dados 'silver' diretamente do banco Postgres.
    Tenta várias tabelas candidatas; se nenhuma existir levantará exceção.
    """
    if engine is None:
        raise ConnectionError("No Postgres engine provided - cannot extract data.")

    df = None
    candidates = [
        "silver.microsoft_security_incident",
        "silver.microsoft_security",
        "silver.security_incident_prediction_silver",
        "microsoft_security_incident",
        "security_incident_prediction_silver"
    ]
    for t in candidates:
        try:
            print(f"Trying to read table: {t}")
            df = pd.read_sql_query(f"SELECT * FROM {t}", con=engine)
            print(f"Read {len(df)} rows from {t}")
            break
        except Exception as e:
            print(f" - couldn't read {t}: {str(e)}")
            continue

    if df is None:
        raise FileNotFoundError(f"No silver table found in Postgres. Tried: {candidates}")

    print("EXTRACT complete. Shape:", df.shape)
    return df


# ---------- TRANSFORM ----------
def transform_to_gold(df: pd.DataFrame) -> pd.DataFrame:
    """
    Transformações típicas para 'gold':
    - normalização de nomes de colunas
    - remoção de colunas com muitos missing (>90%)
    - deduplicação
    - conversão de tipos (tentativa para timestamps e numerics)
    - preenchimento simples para missings
    - seleção heurística de colunas úteis
    """
    print("Starting TRANSFORM...")
    df = df.copy()

    # Normaliza nomes de colunas
    df.columns = [str(c).strip().lower().replace(' ', '_') for c in df.columns]

    # Remove colunas com > 90% de valores ausentes
    missing_ratio = df.isnull().mean()
    to_drop = missing_ratio[missing_ratio > 0.9].index.tolist()
    if to_drop:
        print(f"Dropping columns with >90% missing: {to_drop}")
        df.drop(columns=to_drop, inplace=True)

    # Drop duplicate rows
    before_dup = len(df)
    df.drop_duplicates(inplace=True)
    after_dup = len(df)
    print(f"Dropped {before_dup - after_dup} duplicate rows.")

    # Tenta converter colunas de timestamp automaticamente (nomes comuns)
    ts_candidates = [c for c in df.columns if any(k in c for k in ['timestamp', 'time', 'date', 'created', 'occurred', 'event'])]
    parsed = []
    for c in ts_candidates:
        try:
            df[c] = pd.to_datetime(df[c], errors='coerce', utc=False)
            parsed.append(c)
        except Exception:
            continue
    if parsed:
        print(f"Parsed datetime columns: {parsed}")

    # Converter colunas numéricas (tentativa)
    for c in df.select_dtypes(include=['object']).columns:
        sample = df[c].dropna().astype(str)
        if len(sample) > 0:
            num_like = sample.str.match(r'^-?\d+(\.\d+)?$').mean()
            if num_like > 0.9:
                df[c] = pd.to_numeric(df[c], errors='coerce')

    # Preenchimento simples:
    # numéricos -> mediana; objetos / categóricas -> modo (ou vazio)
    for c in df.columns:
        try:
            if df[c].dtype.kind in 'biufc':  # numeric kinds
                if df[c].isnull().any():
                    med = df[c].median()
                    df[c].fillna(med, inplace=True)
            else:
                if df[c].isnull().any():
                    mode = df[c].mode(dropna=True)
                    if not mode.empty:
                        df[c].fillna(mode.iloc[0], inplace=True)
                    else:
                        df[c].fillna('', inplace=True)
        except Exception:
            df[c].fillna('', inplace=True)

    # Heurística: selecionar colunas que provavelmente importam para um dashboard
    useful_keywords = ['id', 'incident', 'alert', 'timestamp', 'time', 'date', 'severity', 'status',
                       'category', 'description', 'description_text', 'host', 'user', 'org', 'org_id',
                       'source', 'target']
    useful_cols = []
    for kw in useful_keywords:
        useful_cols += [c for c in df.columns if kw in c]
    useful_cols = list(dict.fromkeys(useful_cols))  # preserva ordem e remove duplicatas

    if useful_cols:
        print(f"Selected useful columns (heuristic): {useful_cols}")
        df = df[useful_cols]

    print("TRANSFORM complete. Shape:", df.shape)
    return df


# ---------- LOAD ----------
def load_gold_to_postgres(df: pd.DataFrame, engine, schema: str = 'gold', table_name: str = 'microsoft_security_incident'):
    """
    Grava o dataframe 'gold' no Postgres usando SQLAlchemy.
    Cria o schema se necessário.
    """
    if engine is None:
        raise ConnectionError("No Postgres engine provided - cannot load data.")

    try:
        # criar schema se não existir (executa SQL puro)
        with engine.begin() as conn:
            conn.execute(sqlalchemy.text(f'CREATE SCHEMA IF NOT EXISTS {schema}'))
        # grava a tabela (substitui se já existir)
        df.to_sql(table_name, con=engine, schema=schema, if_exists='replace', index=False, method='multi', chunksize=1000)
        print(f"Wrote GOLD to Postgres table: {schema}.{table_name}")
        return True
    except Exception as e:
        print("Failed to write to Postgres:", e)
        return False


# ---------- Main ETL flow ----------
def run_etl(save_to_postgres: bool = True):
    engine, pg = create_engine_from_compose(repo_root)
    if engine is None:
        print("Cannot proceed: Postgres engine unavailable. Exiting ETL.")
        return

    try:
        raw = extract_silver(engine)
    except Exception as e:
        print("EXTRACT failed:", e)
        return

    gold_df = transform_to_gold(raw)

    if save_to_postgres:
        written = load_gold_to_postgres(gold_df, engine)
        if not written:
            print("Postgres save requested but failed.")
    else:
        print("save_to_postgres=False, skipping writing to Postgres (no CSV fallback).")

    print("ETL finished successfully.")


if __name__ == '__main__':
    run_etl(save_to_postgres=True)


=== ETL Silver -> Gold (EXTRACT / TRANSFORM / LOAD) (Postgres-only) ===
Connected to Postgres: localhost:5433/microsoft-security
Trying to read table: silver.microsoft_security_incident
Read 9516837 rows from silver.microsoft_security_incident
EXTRACT complete. Shape: (9516837, 23)
Starting TRANSFORM...
Dropped 1843021 duplicate rows.
Parsed datetime columns: ['timestamp']
Selected useful columns (heuristic): ['id', 'org_id', 'incident_id', 'alert_id', 'detector_id', 'incident_grade', 'evidence_role', 'device_id', 'account_sid', 'alert_title', 'timestamp', 'category']
TRANSFORM complete. Shape: (7673816, 12)
Wrote GOLD to Postgres table: gold.microsoft_security_incident
ETL finished successfully.


## TRANSFORM

Realizamos transformações: seleção de colunas relevantes, conversão de tipos e criação de features derivadas para dashboards.


### Seleção de Colunas Relevantes para Dashboard

Mantemos apenas colunas úteis para visualizações e agregações em dashboards.


### Conversão de Tipos e Features Derivadas

Convertemos timestamp para datetime e criamos colunas derivadas (ano, mês, dia) para agregações temporais em dashboards.


### Tratamento de Duplicatas e Qualidade Final

Removemos duplicatas e verificamos qualidade.


## LOAD

Carregamos os dados processados para a camada Gold.
